In [1]:
import torchvision
from torchvision import transforms
import torch
from torch.utils.data import DataLoader, random_split
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import copy
import seaborn as sns

torch.manual_seed(42)

Here we select the device to be used for training. We use a GPU if available.

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Download the FashionMNIST Dataset using `torchvision`, transform the dataset to tensor and normalize it from [0,1]

In [5]:
transform = transforms.ToTensor()  # transformer to tensor and normalizes to [0, 1]

train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, 
                                                    download=True, transform=transform)
test_dataset = torchvision.datasets.FashionMNIST(root='./data', train=False, 
                                                   download=True, transform=transform)

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 168kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.14MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 21.4MB/s]


Reserve 10% of the training dataset for the validation dataset.

In [7]:
# Reserve 10% of training set as validation (as per your assignment)
val_size = int(0.1 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_data, val_data = random_split(train_dataset, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

We create a CustomCNN Model below.

In [9]:
class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1) # output dims (32, 28, 28)
        self.maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2) # output dims (32, 14, 14)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1) # output dims (64, 14, 14)
        self.maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2) # output dims (64, 7, 7)
        self.fc1 = nn.Linear(in_features=64*7*7, out_features=128)
        self.fc2 = nn.Linear(in_features=128, out_features=10)
        
    def forward(self, X):
        X = torch.relu(self.conv1(X))
        X = self.maxpool1(X)
        X = torch.relu(self.conv2(X))
        X = torch.flatten(self.maxpool2(X), start_dim=1)
        X = torch.relu(self.fc1(X))
        return self.fc2(X)

In [ ]:
model = CustomCNN().to(device)
lr = 0.001
weight_decay=0.01

optim = torch.optim.AdamW(model.parameters(), lr, weight_decay=weight_decay)
loss = nn.CrossEntropyLoss()
curr_epoch = 1

training_loss = []
validation_loss = []

while True:
    model.train()
    train_loss = 0
    train_total = 0
    train_correct = 0
    
    for data, labels in train_loader:
        # transfer data to the compute device
        images, labels = images.to(device), labels.to(device)
        
        # perform predictions
        output = model(data)
        l = loss(output, labels)
        
        # store data
        predictions = torch.argmax(output)
        train_loss += l.item()
        train_total += labels.size(0) # keep track of the total dataset since you don't know the size of the last batch
        train_correct = (predictions == labels).sum().item()
        
        # apply backpropagtion
        optim.zero_grad()
        l.backward()
        optim.step()
    
    train_avg_loss = len(train_loader)
    train_acc = 

    val_loss = 0
    val_total = 0
    val_correct = 0
    model.eval()
    for data, labels in val_loader:
        data.to(device)
        labels.to(device)
        output = model(data)

        l = loss(output, labels)

        predictions = torch.argmax(output)
        val_loss += l.item()
        val_total += labels.size(0)
        val_correct = (predictions == labels).sum().item()
    
    if ()
    curr_epoch += 1


RuntimeError: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor